# Create an Ensemble of Parameter Files


### **Step 1:** 
The first step in a one-at-a-time sensitivity experiment is to select which parameters you want to modify and what minimum and maximum values to use for each. 
* Here we will start with three CLM-FATES parameters: 
  - `fates_leaf_stomatal_intercept` : *Minimum unstressed stomatal conductance for Medlyn stomatal model* (μmol H<sub>2</sub>0 m<sup>-2</sup> s<sup>-1</sup>)
  - `fates_leaf_vcmax25top` : *maximum carboxylation rate of Rubisco at 25 °C, canopy top* (μmol CO<sub>2</sub> m<sup>-2</sup> s<sup>-1</sup>)
  - `fates_rad_leaf_clumping_index` : *self-occlusion of leaf scattering elements decreasing light interception* (unitless)



CLM and FATES provide suggested ranges for many parameters, but these ranges may not be appropriate for your ecosystem or experiment so you may modify them. In this experiment we will use the default minimum and maximum for each parameter. For more information about the suggested ranges see Kennedy et al., 2025 and Foster et al., 2026. 

Open and inspect the default parameter ranges set in this file:

``meristem/examples/paramgen/fates_example/fates_param_dir/main.csv``

<div class="alert alert-info">
<strong>What are the minimum and maximum settings for the three parameters we've selected?</strong><br><br>
<div>

In [ ]:
import pandas as pd

csv_path = "~/meristem/examples/paramgen/fates_example/fates_param_dir/main.csv"

params = pd.read_csv(csv_path, index_col=0)

param_name = 'fates_rad_leaf_clumping_index'
print(param_name + " minimum = " + str(params.loc[param_name, 'param_min']))
print(param_name + " maximum = " + str(params.loc[param_name, 'param_max']))


: 

Did you notice that the Vcmax25top parameter just says PFT? This is because the ranges differ for each plant functional type. The parameter ranges per pft are in their own csv file: 

In [ ]:
csv_path = "~/meristem/examples/paramgen/fates_example/fates_param_dir/fates_leaf_vcmax25top.csv"
vcmax_pft_params = pd.read_csv(csv_path, index_col=0)
vcmax_pft_params 

: 

### **Step 2:** 

Now we are going to use ``meristem.paramgen`` to generate parameter files for our OAT sensitivity experiment. Given a set of parameters and their ranges, this function produces a collection of parameter files (NetCDF or FATES JSON) that can be fed directly into and ensemble CLM/FATES model runs.

The ``meristem`` packages has a built in sampling design for one-at-a-time minimum maximum sampling. We just have to test the ``ensemble_type`` and provide paths to our parameter information: 

In [ ]:
from meristem.paramgen import ParamEnsemble
 
ensemble = ParamEnsemble.from_dict({
    "ensemble_type": "OAT", # what type of sampling design to use? One-at-a-time (OAT) 
    "param_dir": "fates_param_dir", # Where are the csv files with the parameter information and values stored? 
    "default_param_file": "default_fates_param.json", # What default fates parameter do you want to start from? 
    "ensemble_dir": "paramfiles_OAT", # where do you want the generated parameter files to be stored?
    "file_prefix": "fates_oat", # what should we call your experiment? This will be used as a prefix for the generated parameter files.
    "param_list": ["fates_leaf_stomatal_intercept","fates_leaf_vcmax25top", "fates_rad_leaf_clumping_index"], # which parameters do you want to modify? 
})
 
ensemble.create_ensemble()

: 

Check that the files were created correctly

```
cd ~/meristem/examples/paramgen/fates_example/paramfiles_OAT/
ls
```

You should see 6 paramfiles titled fates_oat_000*.json as well as a "key" telling you which parameter modifications were made in each file: fates_oat_key.csv

In [ ]:
# take a look at the parameter settings in different parameter files. 
import json

file = '~/meristem/examples/paramgen/fates_example/paramfiles_OAT/fates_oat_0000.json'
with open(file) as f:
    params = json.load(f)
params['parameters']['fates_leaf_stomatal_intercept']

In [ ]:
# Look in the key to make sure the parameter perturbations match up: 
csv_path = "~/meristem/examples/paramgen/fates_example/paramfiles_OAT/fates_oat_key.csv"
param_key = pd.read_csv(csv_path, index_col=0)
param_key